# Part 1: Bag of Words Representation

## Read in Data from Demo Code

### Import Notable Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import textwrap

In [80]:
data_dir = 'data_readinglevel'
x_train_df = pd.read_csv(os.path.join(data_dir, 'x_train.csv'))
y_train_df = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))

N, n_cols = x_train_df.shape
print("Shape of x_train_df: (%d, %d)" % (N, n_cols))
print("Shape of y_train_df: %s" % str(y_train_df.shape))

Shape of x_train_df: (5557, 32)
Shape of y_train_df: (5557, 5)


### Test Input Data Parsing

In [81]:
tr_text_list = x_train_df['text'].values.tolist()
prng = np.random.RandomState(101)
rows = prng.permutation(np.arange(y_train_df.shape[0]))
for row_id in rows[:8]:
    text = tr_text_list[row_id]
    print("row %5d | %s BY %s | y = %s" % (
        row_id,
        y_train_df['title'].values[row_id],
        y_train_df['author'].values[row_id],
        y_train_df['Coarse Label'].values[row_id],
        ))
    # Pretty print text via textwrap library
    line_list = textwrap.wrap(tr_text_list[row_id],
        width=70,
        initial_indent='  ',
        subsequent_indent='  ')
    print('\n'.join(line_list))
    print("")

row  4746 | The Red and the Black: A Chronicle of 1830 BY Stendhal | y = Key Stage 4-5
  It was hermetically sealed; he was on the point of  fainting and
  remained for a long time leaning against the oak; then  with a
  staggering step he went to have another look at the gardener's
  ladder. The chain which he had once forced asunder--in, alas, such
  different  circumstances--had not yet been repaired. Carried away by
  a moment of  madness, Julien pressed it to his lips.

row  1250 | Cranford BY Elizabeth Cleghorn Gaskell | y = Key Stage 4-5
  Miss Pole, Miss Matty, and I, meanwhile attended to Miss Brown: and
  hard  work we found it to relieve her querulous and never-ending
  complaints. But if we were so weary and dispirited, what must Miss
  Jessie have been! Yet she came back almost calm as if she had gained
  a new strength. She  put off her mourning dress, and came in,
  looking pale and gentle,  thanking us each with a soft long pressure
  of the hand.

row    54 | The Big F

## Part 1: Cleaning CSV File into a list of Tokens

We first define the our tokenize_text function

In [4]:
def tokenize_text(raw_text):
    ''' Transform a plain-text string into a list of tokens
    
    We assume that *whitespace* divides tokens.
    
    Args
    ----
    raw_text : string
    
    Returns
    -------
    list_of_tokens : list of strings
        Each element is one token in the provided text
    '''
    list_of_tokens = raw_text.split() # split method divides on whitespace by default
    for pp in range(len(list_of_tokens)):
        cur_token = list_of_tokens[pp]
        # Remove punctuation
        for punc in ['?', '!', '_', '.', ',', '"', '/']:
            cur_token = cur_token.replace(punc, "")
        # Turn to lower case
        clean_token = cur_token.lower()
        # Replace the cleaned token into the original list
        list_of_tokens[pp] = clean_token
    return list_of_tokens

### Testing our Tokenize Function on the tr_test_list:

In [5]:
for line in tr_text_list[:10]:
    print("\nRaw text:")
    print(line)
    print("Clean token list:")
    print(tokenize_text(line))


Raw text:
"Yes.... What sort of terms was he on with the guests—you and Miss  Norris and all of them?" "Just polite and rather silent, you know. Keeping himself to himself. We didn't see so very much of him, except at meals. We were here to  enjoy ourselves, and—well, _he_ wasn't." "He wasn't there when the ghost walked?" "No. I heard Mark calling for him when he went back to the house. I  expect Cayley stroked down his feathers a bit, and told him that girls  will be girls....—Hallo, here we are."
Clean token list:
['yes', 'what', 'sort', 'of', 'terms', 'was', 'he', 'on', 'with', 'the', 'guests—you', 'and', 'miss', 'norris', 'and', 'all', 'of', 'them', 'just', 'polite', 'and', 'rather', 'silent', 'you', 'know', 'keeping', 'himself', 'to', 'himself', 'we', "didn't", 'see', 'so', 'very', 'much', 'of', 'him', 'except', 'at', 'meals', 'we', 'were', 'here', 'to', 'enjoy', 'ourselves', 'and—well', 'he', "wasn't", 'he', "wasn't", 'there', 'when', 'the', 'ghost', 'walked', 'no', 'i', 'heard'

## Part 2: Defining a Fixed Size Vocabulary 

Firstly, count the number of instances a particular token shows up:

In [6]:
tok_count_dict = dict()

for line in tr_text_list:
    tok_list = tokenize_text(line)
    for tok in tok_list:
        if tok in tok_count_dict:
            tok_count_dict[tok] += 1
        else:
            tok_count_dict[tok] = 1

Print 10 most common tokens:

In [7]:
sorted_tokens = list(sorted(tok_count_dict, key=tok_count_dict.get, reverse=True))

print("Top 10 most frequent")
for w in sorted_tokens[:10]:
    print("%5d %s" % (tok_count_dict[w], w))

print("10 Least Frequent:\n")
for w in sorted_tokens[-10:]:
    print("%5d %s" % (tok_count_dict[w], w))

Top 10 most frequent
24173 the
14483 and
12029 of
11903 to
 9227 a
 6778 in
 6728 i
 5982 he
 5501 that
 5258 was
10 Least Frequent:

    1 honorable;
    1 saucy
    1 brandied
    1 absinthe
    1 teased
    1 coquenard
    1 madinier
    1 hearse
    1 lumpy
    1 monumental


## Part 3: Creating Bag of Words representation

Define a dictionary that maps each vocab word to an integer defining its ordering in the vocabulary set

In [8]:
# For now we're not removing any words for the sake of using BoW size as a hyper parameter, so sorted_tokens is just entire tokens array

vocab_dict = dict()
for vocab_id, tok in enumerate(sorted_tokens):
    vocab_dict[tok] = vocab_id

## Part 4: Using CountVectorizer and Logistic Regression for Binary Classification

Import the library:

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

### Create a Counter Vectorizer that uses our vocabulary list:

In [10]:
bow_preprocessor = CountVectorizer(binary=False, vocabulary=vocab_dict)

bow_preprocessor.fit(tr_text_list)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


Then we use scikit `transform` to have a sparse matrix transformation to obtain the features:

In [11]:
sparse_arr = bow_preprocessor.transform(tr_text_list)
print(type(sparse_arr))
print(sparse_arr.shape)

# Convert to a dense representation

dense_arr_3V = sparse_arr.toarray()
print(type(dense_arr_3V))
print(dense_arr_3V.shape)

dense_arr_3V

<class 'scipy.sparse._csr.csr_matrix'>
(5557, 34884)
<class 'numpy.ndarray'>
(5557, 34884)


array([[3, 5, 3, ..., 0, 0, 0],
       [4, 0, 1, ..., 0, 0, 0],
       [6, 2, 2, ..., 0, 0, 0],
       ...,
       [4, 1, 1, ..., 0, 0, 0],
       [3, 3, 1, ..., 1, 1, 0],
       [6, 3, 1, ..., 0, 0, 1]], shape=(5557, 34884))

### CountVectorizer that builds its own UNIGRAM vocabulary

* ngram_range=(1,1) : Means it will use unigrams (individual tokens)
* min_df=1 : Means include any term that occurs in at least one document in training set
* max_df=1.0 : Means include any terms that appears in less than 100% (fraction of 1.0) of training docs
* binary=False : means it will do counts, not just binary presence/absence of each vocab term


In [12]:
bow_preprocessor = CountVectorizer(ngram_range=(1,1), min_df=1, max_df=1.0, binary=False)

In [13]:
bow_preprocessor.fit(tr_text_list)
len(bow_preprocessor.vocabulary_)

26866

Print first 10 words in vocabulary for testing

In [14]:
for term, count in list(bow_preprocessor.vocabulary_.items())[:10]:
    print("%4d %s" % (count, term))

26745 yes
26144 what
22259 sort
16713 of
23784 terms
25960 was
11530 he
16804 on
26418 with
23847 the


## Part 5: Bag of Words Count Vectorizer + Classifier Pipeline

### Create a pipeline + Classifier

In [53]:
import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score


Print Statements Just to make sure I understand the Text List indexing:

In [22]:
print("This is the first entry: ", tr_text_list[0])
print("This is the second entry: ", tr_text_list[1])

This is the first entry:  "Yes.... What sort of terms was he on with the guests—you and Miss  Norris and all of them?" "Just polite and rather silent, you know. Keeping himself to himself. We didn't see so very much of him, except at meals. We were here to  enjoy ourselves, and—well, _he_ wasn't." "He wasn't there when the ghost walked?" "No. I heard Mark calling for him when he went back to the house. I  expect Cayley stroked down his feathers a bit, and told him that girls  will be girls....—Hallo, here we are."
This is the second entry:  Perhaps I should say that it was Mark's  private plan. My own was different. "The announcement at breakfast went well. After the golfing-party had  gone off, we had the morning in which to complete our arrangements. What I was chiefly concerned about was to establish as completely as  possible the identity of Robert.


Creating the proper training and testing datasets:

In [52]:
# Create the binary classification column y
# 2-3 is labeled as 0
# 4-5 is labeled as 1

y_train_df
y_train_df["true_labels"] = np.where(y_train_df["Coarse Label"].str.contains("Key Stage 2-3"), 0, 1)
y = y_train_df.true_labels.to_numpy() 


"y" is now the list of all the correct labels for the training data

"tr_text_list" is now the input list of all the training data text

Full Pipeline, Data Splitting, Paramter Search, and Cross Validation: 

In [54]:
# split the data into training and testing
X_train, X_test, y_train, y_test = train_test_split(tr_text_list,y, test_size = 0.2, random_state=42, stratify=y)

In [55]:
# define the pipeline
my_bow_classifier_pipeline = sklearn.pipeline.Pipeline([
    ('my_bow_feature_extractor', CountVectorizer(ngram_range=(1,1), lowercase= True)),
    ('my_classifier', sklearn.linear_model.LogisticRegression(max_iter=1000, random_state=101)),
])

In [56]:
# define the parameter grid
param_grid = {
    'my_bow_feature_extractor__min_df': [1,2,5,10],
    'my_bow_feature_extractor__max_df': [0.85, 0.9, 0.95, 1.0],
    'my_classifier__C': np.logspace(-4, 4, 9)
}

In [59]:
# grid search with 5 folds
# Score with area under ROC

grid = GridSearchCV(
    my_bow_classifier_pipeline,
    param_grid,
    cv=5,
    scoring='roc_auc'
)

In [60]:
# Fit on the training data
grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._state=101))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'my_bow_feature_extractor__max_df': [0.85, 0.9, ...], 'my_bow_feature_extractor__min_df': [1, 2, ...], 'my_classifier__C': array([1.e-04... 1.e+04])}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computatio

In [68]:
# Print out the results
# Best parameters
print("Best parameters:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)

Best parameters: {'my_bow_feature_extractor__max_df': 0.95, 'my_bow_feature_extractor__min_df': 1, 'my_classifier__C': np.float64(1.0)}
Best CV ROC-AUC: 0.8061924101474908


In [69]:
# Evaluate on test set (final unbiased score)
y_pred_proba = grid.best_estimator_.predict_proba(X_test)[:, 1]
print("Test ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

Test ROC-AUC: 0.8018287505714845


SEE WHICH WORDS MATTERED:


In [77]:
best_model = grid.best_estimator_
vectorizer = best_model.named_steps['my_bow_feature_extractor']
classifier = best_model.named_steps['my_classifier']

feature_names = vectorizer.get_feature_names_out()
weights = classifier.coef_[0]  # binary classification

array([-0.08419566,  0.00289981, -0.08352587, ...,  0.03300188,
        0.00404405,  0.01279157], shape=(24183,))

In [78]:
# Pair words with weights
coef_df = pd.DataFrame({
    "word": feature_names,
    "weight": weights
}).sort_values("weight", ascending=False)

print("\nTop positive words:")
print(coef_df.head(20))

print("\nTop negative words:")
print(coef_df.tail(20))


Top positive words:
            word    weight
12136     julien  1.427513
22126      trina  1.327589
12153     jurgis  1.188565
16002   peterkin  1.118421
1576         ann  1.029142
10974    husband  0.950568
10749     honour  0.925379
23966   wretched  0.912526
16236      plain  0.896251
9515    gervaise  0.875903
23761     winter  0.860057
13556     martin  0.845844
20037     sought  0.841916
21519  therefore  0.835233
4181     charles  0.830971
22717       unto  0.826976
1974      asking  0.809130
1929      arthur  0.797246
7553       emily  0.787357
5987       death  0.786896

Top negative words:
             word    weight
1223        agnes -0.940553
14004     minutes -0.945393
10409  heathcliff -0.950057
1785       aramis -1.010898
21220        talk -1.013927
7488       elnora -1.038755
17473          re -1.056915
5940       davies -1.115319
13610       match -1.119784
2374        bambi -1.128447
1860       arkady -1.139129
1924     artagnan -1.161374
14978     oblomov -1.184415

PREDICT ON THE REAL TEST SET:

In [82]:
# Load the test data
x_test_df = pd.read_csv(os.path.join(data_dir, 'x_test.csv'))
te_text_list = x_test_df['text'].values.tolist()

y_proba1_test = grid.best_estimator_.predict_proba(te_text_list)[:, 1]

In [88]:
print(y_proba1_test)
np.savetxt('yproba1_test.txt', y_proba1_test)

[0.93303811 0.99845273 0.95543657 ... 0.89086903 0.63257695 0.85240712]
